# 模型调用

In [188]:
import os

from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage


def _call_chat_model(
    *,
    model_name: str,
    base_url: str,
    api_key: str | None,
    system_prompt: str,
    human_message: str,
) -> str:
    model = init_chat_model(
        model=model_name, base_url=base_url, api_key=api_key, model_provider="openai", temperature=0.7
    )
    response = model.invoke([SystemMessage(content=system_prompt), HumanMessage(content=human_message)])
    return response.content.strip()


def deepseek_chat(prompt: str, model_name: str = "deepseek-chat") -> str:
    return _call_chat_model(
        model_name=model_name,
        base_url="https://api.deepseek.com/v1",
        api_key=os.getenv("DEEPSEEK_API"),
        system_prompt="你是任务分析助手，请用简明中文解释用户的问题并规划回答结构",
        human_message=prompt,
    )


# 此处用 DeepSeek 模拟
def qwen_chat(prompt: str, model_name: str = "deepseek-chat") -> str:
    return _call_chat_model(
        model_name=model_name,
        base_url="https://api.deepseek.com/v1",
        api_key=os.getenv("DEEPSEEK_API"),
        system_prompt="你是专业的知识问答助手， 请根据提供的分析和上下文生成详细、准确的中文答案。",
        human_message=prompt,
    )


## 测试 deepseek_chat

In [189]:
deepseek_chat("规划去新都桥的旅程")

'好的，我来解释您的问题并规划回答结构。\n\n**问题解释**：  \n您希望规划一次前往四川省甘孜藏族自治州新都桥镇的旅行。新都桥是川藏线（318国道）上的重要节点，以“摄影天堂”著称，秋季风景尤为优美。您需要一份涵盖交通、景点、住宿、时间安排等内容的实用旅行规划。\n\n**回答结构规划**：  \n1. **最佳旅行时间**：推荐秋季（10月中下旬）或夏季（7-8月），说明理由。  \n2. **交通方式**：  \n   - 自驾：从成都出发，经雅康高速、318国道，约5-6小时。  \n   - 班车：成都新南门车站有到康定的大巴，再转车到新都桥。  \n3. **核心景点**：  \n   - 塔公草原、塔公寺  \n   - 墨石公园  \n   - 居里寺、贡嘎雪山观景台  \n   - 沿途风光（折多山、雅拉雪山）  \n4. **行程建议**：  \n   - 2日游：第一天抵达、周边游玩；第二天深度探索或返程。  \n   - 3日游：增加塔公、墨石公园等远途景点。  \n5. **住宿与美食**：  \n   - 推荐藏式民宿、青旅或酒店。  \n   - 特色饮食：酥油茶、牦牛肉、青稞饼。  \n6. **注意事项**：  \n   - 高原反应预防（海拔约3300米）。  \n   - 天气多变，备好衣物、防晒和氧气。  \n   - 尊重当地风俗（如寺庙礼仪）。  \n7. **预算参考**：  \n   - 交通、住宿、餐饮、门票的大致费用范围。  \n\n接下来，我将按此结构为您详细展开规划。'

## 测试 qwen_chat

In [190]:
qwen_chat("什么是量子力学？")

'量子力学是物理学的一个基础分支，研究微观粒子（如原子、电子、光子等）的行为规律。它描述的是在极小的尺度上，物质和能量的性质如何与经典物理（如牛顿力学）截然不同。量子力学的核心概念包括：\n\n1. **波粒二象性**：微观粒子既表现出粒子性（如位置、动量），又表现出波动性（如干涉、衍射）。例如，光可以表现为光子（粒子），也可以表现为电磁波。\n2. **不确定性原理**：由海森堡提出，指出无法同时精确测量一对共轭物理量（如位置和动量）。测量越精确的位置，动量的不确定性就越大。\n3. **量子态与叠加**：系统状态由波函数描述，波函数可以叠加，意味着粒子可以同时处于多个状态（如薛定谔的猫思想实验中的死与活并存），直到测量时才“坍缩”为一个确定状态。\n4. **量子纠缠**：两个或多个粒子可以相互关联，无论距离多远，对其中一个粒子的测量会瞬间影响另一个粒子的状态。这挑战了经典物理的局域实在性，并被用于量子通信和量子计算。\n5. **能级量子化**：原子中的电子只能处于特定的能量状态（能级），跃迁时吸收或发射特定能量的光子。\n\n量子力学的应用广泛，包括半导体技术（晶体管、芯片）、激光、核磁共振成像、量子计算、密码学以及解释化学键和物质性质。它由普朗克、爱因斯坦、玻尔、薛定谔、海森堡等科学家在20世纪初建立，至今仍是现代物理学的基石之一。'

# 工具

In [191]:
from typing import Any


def _ensure_question_text(raw: Any) -> str:
    if isinstance(raw, str):
        return raw
    """提取问题文本"""
    if isinstance(raw, str):
        return raw

    if isinstance(raw, dict):
        for key in ("question", "topic", "prompt"):
            value = raw.get(key)
            if isinstance(value, str) and value.strip():
                return value
        joined = " ".join(str(v) for v in raw.values())
        if joined.strip():
            return joined

    return str(raw)

## 测试

In [192]:
_ensure_question_text("123")

'123'

In [193]:
_ensure_question_text({"a": 1, "b": 2})

'1 2'

# 状态

In [194]:
import operator
from dataclasses import dataclass
from typing import Annotated, List, NotRequired, TypedDict

from pydantic import BaseModel, Field


@dataclass
class AssistantContext:
    analysis_model: str = "deepseek-chat"
    answer_model: str = "qwen-plus"
    detail_style: str = "auto"  # auto | summary | deep


class QAState(TypedDict):
    """问答状态"""

    question: str  # 用户输入
    analysis: Annotated[List[str], operator.add]  # 分析结果
    plan: Annotated[List[str], operator.add]  # 回答计划
    draft_sections: Annotated[List[str], operator.add]  # 章节草稿
    answer: Annotated[List[str], operator.add]  # 最终回答
    need_summarize: bool  # 是否需要总结
    expected_sections: int  # 章节数
    completed_sections: Annotated[int, operator.add]  # 已完成章节数
    ready_for_summary: bool  # 是否可总结
    current_section: NotRequired[str]  # 当前章节标题


# class QAState(BaseModel):
#     question: str = Field(description="用户输入")
#     analysis: Annotated[List[str], operator.add] = Field(description="分析结果")
#     plan: Annotated[List[str], operator.add] = Field(description="回答计划")
#     draft_sections: Annotated[List[str], operator.add] = Field(description="章节草稿")
#     answer: Annotated[List[str], operator.add] = Field(description="最终答案")
#     need_summarize: bool = Field(description="是否需要总结")
#     expected_section: int = Field(description="章节数")
#     completed_section: Annotated[int, operator.add] = Field(description="已完成章节数")
#     ready_for_summary: bool = Field(description="是否可总结")
#     current_section: str | None = Field(description="当前章节")

# 节点

In [195]:
from langgraph.func import task, entrypoint
from langgraph.checkpoint.memory import InMemorySaver


@task
def split_question(question: str) -> List[str]:
    """拆分问题为多个关注点"""
    question_text = _ensure_question_text(question)
    normalized = question_text
    # 标点替换，可是这里替换了逻辑词
    for sep in ["，", "。", "？", "、", ";", "；", ",", "?", "以及", "并且", "和", "与"]:
        normalized = normalized.replace(sep, "|")

    segments = [seg.strip() for seg in normalized.split("|") if seg.strip()]
    #
    if not segments and question_text.strip():
        segments = [question_text.strip()]
    r = segments[:5] if segments else ["核心概念", "关键要点", "典型应用"]
    print(f"split_question: {r}")
    return r


@task
def enrich_outline(segments: List[str]) -> List[str]:
    """生成章节标题"""
    outline = []
    for idx, seg in enumerate(segments, 1):
        outline.append(f"第{idx}节：{seg}")
    print(f"enrich_outline: {outline}")
    return outline


@entrypoint(checkpointer=InMemorySaver())
def planning_workflow(question: str):
    """规划工作流"""
    question_text = _ensure_question_text(question)
    segments = split_question(question_text).result()
    outline = enrich_outline(segments).result()
    r = {"steps": outline}
    print(f"planning_workflow: {r}")
    return r

# 节点

In [196]:
from typing import Dict

from langgraph.runtime import Runtime
from langgraph.types import Send


def analyze_question(state: QAState, runtime: Runtime[AssistantContext]) -> Dict[str, Any]:
    """分析问题并制定计划"""
    question = state["question"]
    detail_pref = runtime.context.detail_style

    if detail_pref == "summary":
        need_summarize = True
    elif detail_pref == "deep":
        need_summarize = False
    else:
        need_summarize = not any(k in question for k in ["详细", "深度", "长篇", "展开"])

    content = deepseek_chat(
        f"请分析以下问题的主题、信息类型，并给出回答思路：{question}",
        model_name=runtime.context.analysis_model,
    )
    r = {
        "analysis": [content],
        "need_summarize": need_summarize,
        "completed_sections": 0,
        "ready_for_summary": need_summarize,
    }
    print(f"analyze_question: {r}")
    return r


def plan_with_functional_api(state: QAState) -> Dict[str, Any]:
    """调用规划工作流"""
    workflow_result = planning_workflow.invoke({"question": state["question"]})
    outline = workflow_result.get("steps", [])
    if not outline:
        outline = ["第1节：核心概念速览", "第2节：关键能力", "第3节：企业落地建议"]
    r = {
        "plan": outline,
        "analysis": [f"规划输出 {len(outline)} 个章节"],
        "expected_sections": len(outline),
    }
    print(f"plan_with_functional_api: {r}")
    return r


def dispatch_sections(state: QAState) -> Dict[str, Any]:
    """章节分发节点"""
    print("dispatch_sections: {}")
    return {}


def fan_out_sections(state: QAState):
    """并行发送章节任务"""
    sections = state.get("plan", [])
    sends = [
        Send("expand_section", {"question": state["question"], "current_section": section}) for section in sections
    ]
    sends.append(Send("collect_sections", {}))
    print(f"fan_out_sections: {sends}")
    return sends


def expand_section(state: QAState, runtime: Runtime[AssistantContext]) -> Dict[str, Any]:
    """生成单个章节内容"""

    section = state.get("current_section", "章节")
    prompt = f"""
        用户问题：
        {state["question"]}

        分析依据：
        {"".join(state.get("analysis", []))}

        当前需要展开的章节：
        {section}

        已有章节草稿：
        {" | ".join(state.get("draft_sections", [])) or "（暂无）"}

        请输出本章节的核心内容，强调与原问题的关联。
    """
    detailed = qwen_chat(prompt, model_name=runtime.context.answer_model)
    r = {
        "draft_sections": [f"{section}\n{detailed}"],
        "completed_sections": 1,
    }
    print(f"expand_section: {r}")
    return r


def collect_sections(state: QAState) -> Dict[str, Any]:
    """收集章节结果"""
    expected = state.get("expected_sections", 0)
    completed = state.get("completed_sections", 0)
    r = {"ready_for_summary": True} if expected == 0 or completed >= expected else {}
    print(f"collect_sections: {r}")
    return r


def summarize_answer(state: QAState, runtime: Runtime[AssistantContext]) -> Dict[str, Any]:
    """生成最终回答"""
    if not state["need_summarize"] and not state.get("ready_for_summary", False):
        return {}

    prompt = f"""
        用户问题：
        {state["question"]}

        问题分析：
        {"\n".join(state.get("analysis", []))}

        章节规划：
        {"; ".join(state.get("plan", [])) or "（无显式章节）"}

        章节草稿：
        {"\n\n".join(state.get("draft_sections", [])) or "（已启用简要模式）"}

        请基于以上上下文，输出结构化、清晰且自然的中文最终回答。
    """
    print("问题分析：", state.get("analysis", []))
    answer = qwen_chat(prompt, model_name=runtime.context.answer_model)
    r = {"answer": [answer]}
    print(f"summarize_answer: {r}")
    return r

# 条件边

In [197]:
from typing import Literal


def route_after_plan(state: QAState) -> Literal["dispatch", "summarize"]:
    """路由决策"""
    r = "summarize" if state.get("need_summarize") else "dispatch"

    print(state["analysis"])

    print(f"route_after_plan: {r}")
    return r

# 创建图

In [198]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(QAState, context_schema=AssistantContext)

# 节点注册
graph.add_node("analyze", analyze_question)
graph.add_node("planner", plan_with_functional_api)
graph.add_node("dispatch", dispatch_sections)
graph.add_node("expand_section", expand_section)
graph.add_node("collect_sections", collect_sections)
graph.add_node("summarize", summarize_answer)

# 边定义
graph.add_edge(START, "analyze")
graph.add_edge("analyze", "planner")
graph.add_conditional_edges("planner", route_after_plan, {"dispatch": "dispatch", "summarize": "summarize"})
graph.add_conditional_edges("dispatch", fan_out_sections)
graph.add_edge("expand_section", "collect_sections")
graph.add_edge("collect_sections", "summarize")
graph.add_edge("summarize", END)

# 编译图
app = graph.compile()

# 执行

In [199]:
import json


def create_initial_state(question: str) -> QAState:
    """创建初始状态"""
    return {
        "question": question,
        "analysis": [],
        "plan": [],
        "draft_sections": [],
        "answer": [],
        "need_summarize": False,
        "expected_sections": 0,
        "completed_sections": 0,
        "ready_for_summary": False,
    }


# user_input = input("请输入一个问题：").strip()
initial_state = create_initial_state("什么是langchain?什么是state?")
# print(json.dumps(initial_state, indent=4))
runtime_context = {
    "analysis_model": "deepseek-chat",
    "answer_model": "deepseek-chat",
    "detail_style": "deep",
}

print("\n=== LangGraph 智能问答助理启动 ===\n")
print(f"运行时上下文：{runtime_context}")

result = app.invoke(initial_state, context=runtime_context)
print("\n--- result全貌 ---")
print(type(result))
print(json.dumps(result, indent=4))

print("\n--- 最终输出 ---")
for msg in result["answer"]:
    print(msg)


=== LangGraph 智能问答助理启动 ===

运行时上下文：{'analysis_model': 'deepseek-chat', 'answer_model': 'deepseek-chat', 'detail_style': 'deep'}
analyze_question: {'analysis': ['好的，我们来分析一下您的问题。\n\n**主题：**\n您的问题是关于 **LangChain** 和 **State** 这两个核心概念的定义。具体来说，是询问它们在（通常指AI应用开发，特别是大语言模型应用）这个特定领域中的含义。\n\n**信息类型：**\n这是一个**概念解释型**问题。用户需要清晰、准确的定义，并可能希望理解这两个概念之间的关系。\n\n**回答结构规划：**\n\n1.  **LangChain 的定义与核心作用**\n    *   **简要定义：** LangChain 是一个用于开发由大语言模型（LLM）驱动的应用程序的开源框架。\n    *   **核心作用：** 它解决了将LLM与外部数据、工具（如搜索引擎、API、数据库）以及多步骤逻辑（链、代理）连接起来的复杂性问题。可以把它理解为一个“LLM应用的脚手架”或“胶水”。\n    *   **关键特性（可选，视情况展开）：** 链（Chains）、代理（Agents）、记忆（Memory）、文档加载器、提示模板。\n\n2.  **State 的定义与在LangChain中的含义**\n    *   **通用定义：** 在编程中，State（状态）指系统在某个时刻的**快照**或**情况**，包括所有相关的数据（变量值、上下文等）。\n    *   **在LangChain中的具体含义：**\n        *   **在代理（Agent）中：** State 通常指**中间步骤的状态**。例如，代理在决定下一步行动前，需要知道它已经执行了哪些步骤、得到了哪些结果、当前还缺少什么信息。\n        *   **在链（Chain）中/对话记忆（Memory）中：** State 指**对话历史**。即之前的所有用户输入和AI回复，用于保持上下文连贯性。\n        *   **在LangGraph（LangChain的高级扩展）中：** S